# Dataset

In [ ]:
!kaggle dataset download -d masoudnickparvar/brain-tumor-mri-dataset

In [ ]:
!unzip brain-tumor-mri-dataset.zip -d brain-tumor-mri-dataset

# RadImageNet

## tes classifier

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DATA_DIR = "/content/brain-tumor-mri-dataset"

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Training"),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Testing"),
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

print("Classes:", train_dataset.classes)
print("Train :", len(train_dataset))
print("Test  :", len(test_dataset))

# RadImageNet weight from labrasool
model_path = hf_hub_download(
    repo_id="Lab-Rasool/RadImageNet",
    filename="DenseNet121.pt"
)

state_dict = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

# DenseNet121 architecture
model = models.densenet121(weights=None)

new_state_dict = {}
for key, value in state_dict.items():
    if key.startswith('backbone.0.'):
        new_key = key.replace('backbone.0.', 'features.')
        new_state_dict[new_key] = value
    else:
        new_state_dict[key] = value

# Load weights
model.load_state_dict(new_state_dict, strict=False)

model.classifier = nn.Linear(
    model.classifier.in_features,
    4
)

for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)
print("Classifier:", model.classifier)

# test
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input :", images.shape)
print("Output:", outputs.shape)
print("Labels:", labels.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())
print("all good")

In [ ]:
import time

epochs = 50
print(f"training {epochs} epoch")
start_time = time.time()

model.train()
for epoch in range(epochs):
    epoch_start = time.time()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / len(train_loader)

    # progres per 5 epoch
    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | ET: {epoch_time:.2f}s")

total_time = time.time() - start_time
print(f"\n{epochs} epoch done")
print(f"Total ET: {total_time:.2f}s ({total_time/60:.2f} mins)")
print(f"avg ET per epoch: {total_time/epochs:.2f}s")

## tes 1 block

In [ ]:
import time, torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download
from collections import OrderedDict

DATA_DIR = "/content/brain-tumor-mri-dataset"
BATCH_SIZE, EPOCHS, LR = 32, 10, 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/Training", tfm)
test_ds = datasets.ImageFolder(f"{DATA_DIR}/Testing", tfm)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

path = hf_hub_download("Lab-Rasool/RadImageNet", "DenseNet121.pt")
sd = torch.load(path, map_location="cpu", weights_only=True)
sd = OrderedDict(
    (k.replace("backbone.0.", "features.", 1), v)
    for k, v in sd.items() if k.startswith("backbone.0.")
)

model = models.densenet121(weights=None)
result = model.load_state_dict(sd, strict=False)

missing = [k for k in result.missing_keys if not k.startswith("classifier.")]
if missing:
    raise RuntimeError(f"missing: {missing[:10]}")

model.classifier = nn.Linear(model.classifier.in_features, len(train_ds.classes))

# freezing param 
for p in model.parameters():
    p.requires_grad = False

# unfreeze param
for layer in (model.features.denseblock4,
              model.features.norm5,
              model.classifier):
    for p in layer.parameters():
        p.requires_grad = True

model = model.to(device)
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad), lr=LR
)
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Trainable params:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

for epoch in range(EPOCHS):
    model.train()
    start = time.perf_counter()
    loss_sum, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    print(f"Epoch {epoch+1} | Loss: {loss_sum/total:.4f} | "
          f"Accuracy: {correct/total:.4f} | "
          f"Training time: {time.perf_counter()-start:.2f}s")

    model.eval()

## tes 2 block

In [ ]:
import time, torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download
from collections import OrderedDict

DATA_DIR = "/content/brain-tumor-mri-dataset"
BATCH_SIZE, EPOCHS, LR = 32, 10, 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/Training", tfm)
test_ds = datasets.ImageFolder(f"{DATA_DIR}/Testing", tfm)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

path = hf_hub_download("Lab-Rasool/RadImageNet", "DenseNet121.pt")
sd = torch.load(path, map_location="cpu", weights_only=True)
sd = OrderedDict(
    (k.replace("backbone.0.", "features.", 1), v)
    for k, v in sd.items() if k.startswith("backbone.0.")
)

model = models.densenet121(weights=None)
result = model.load_state_dict(sd, strict=False)

missing = [k for k in result.missing_keys if not k.startswith("classifier.")]
if missing:
    raise RuntimeError(f"Backbone tidak termuat lengkap: {missing[:10]}")

model.classifier = nn.Linear(model.classifier.in_features, len(train_ds.classes))

# freezing param 
for p in model.parameters():
    p.requires_grad = False

# unfreeze param
for layer in (model.features.denseblock3,
              model.features.transition3,
              model.features.denseblock4,
              model.features.norm5,
              model.classifier):
    for p in layer.parameters():
        p.requires_grad = True

model = model.to(device)
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad), lr=LR
)
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Trainable params:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

for epoch in range(EPOCHS):
    model.train()
    start = time.perf_counter()
    loss_sum, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    print(f"Epoch {epoch+1} | Loss: {loss_sum/total:.4f} | "
          f"Accuracy: {correct/total:.4f} | "
          f"Training time: {time.perf_counter()-start:.2f}s")

    model.eval()

## full ft

In [ ]:
import os, time, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from huggingface_hub import hf_hub_download
from collections import OrderedDict
import torch
import torch.nn as nn
from torchvision import models

DATA_DIR = "/content/brain-tumor-mri-dataset"
BATCH_SIZE, EPOCHS, LR = 32, 10, 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/Training", transform)
test_ds  = datasets.ImageFolder(f"{DATA_DIR}/Testing", transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

path = hf_hub_download(
    repo_id="Lab-Rasool/RadImageNet",
    filename="DenseNet121.pt"
)
state_dict = torch.load(path, map_location="cpu", weights_only=True)

# replace "backbone.0." from labrasool with "features."  
state_dict = OrderedDict(
    (k.replace("backbone.0.", "features.", 1), v)
    for k, v in state_dict.items()
    if k.startswith("backbone.0.")
)

model = models.densenet121(weights=None)
result = model.load_state_dict(state_dict, strict=False)

missing_backbone = [
    k for k in result.missing_keys
    if not k.startswith("classifier.")
]

if missing_backbone:
    raise RuntimeError(f"missing: {missing_backbone[:10]}")

model.classifier = nn.Linear(model.classifier.in_features, 4)
model = model.to(device)

print("Classifier:", model.classifier)
print("Unexpected keys:", result.unexpected_keys)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Train samples:", len(train_ds), "| Test samples:", len(test_ds))
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(EPOCHS):
    model.train()
    start = time.perf_counter()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    train_seconds = time.perf_counter() - start

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/total:.4f} | "
        f"Accuracy: {correct/total:.4f} | "
        f"Training time: {train_seconds:.2f}s"
    )

# ImageNet

## tes classifier

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

DATA_DIR = "/content/brain-tumor-mri-dataset"

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Training"),
    transform=transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "Testing"),
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

print("Classes:", train_dataset.classes)
print("Train :", len(train_dataset))
print("Test  :", len(test_dataset))

# DenseNet121 architecture
model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

model.classifier = nn.Linear(
    model.classifier.in_features,
    4
)

for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)
print("Classifier:", model.classifier)

# test
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input :", images.shape)
print("Output:", outputs.shape)
print("Labels:", labels.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())
print("all good")

In [ ]:
import time
print("Trainable params:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
epochs = 10
print(f"training {epochs} epoch")
start_time = time.time()

model.train()
for epoch in range(epochs):
    epoch_start = time.time()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / len(train_loader)

    # progres per 5 epoch
    if (epoch + 1) % 5 == 0 or epoch == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | ET: {epoch_time:.2f}s")

total_time = time.time() - start_time
print(f"\n{epochs} epoch done")
print(f"Total ET: {total_time:.2f}s ({total_time/60:.2f} mins)")
print(f"avg ET per epoch: {total_time/epochs:.2f}s")

## tes 1 block

In [ ]:
import time, torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from collections import OrderedDict

DATA_DIR = "/content/brain-tumor-mri-dataset"
BATCH_SIZE, EPOCHS, LR = 32, 10, 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/Training", tfm)
test_ds = datasets.ImageFolder(f"{DATA_DIR}/Testing", tfm)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)


model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

model.classifier = nn.Linear(model.classifier.in_features, len(train_ds.classes))

# freezing param
for p in model.parameters():
    p.requires_grad = False

# unfreeze param
for layer in (model.features.denseblock4,
              model.features.norm5,
              model.classifier):
    for p in layer.parameters():
        p.requires_grad = True

model = model.to(device)
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad), lr=LR
)
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Trainable params:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

for epoch in range(EPOCHS):
    model.train()
    start = time.perf_counter()
    loss_sum, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    print(f"Epoch {epoch+1} | Loss: {loss_sum/total:.4f} | "
          f"Accuracy: {correct/total:.4f} | "
          f"Training time: {time.perf_counter()-start:.2f}s")

## tes 2 block

In [ ]:
import time, torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from huggingface_hub import hf_hub_download
from collections import OrderedDict

DATA_DIR = "/content/brain-tumor-mri-dataset"
BATCH_SIZE, EPOCHS, LR = 32, 10, 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/Training", tfm)
test_ds = datasets.ImageFolder(f"{DATA_DIR}/Testing", tfm)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

model.classifier = nn.Linear(model.classifier.in_features, len(train_ds.classes))

# freezing param
for p in model.parameters():
    p.requires_grad = False

# unfreeze param
for layer in (model.features.denseblock3,
              model.features.transition3,
              model.features.denseblock4,
              model.features.norm5,
              model.classifier):
    for p in layer.parameters():
        p.requires_grad = True

model = model.to(device)
optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad), lr=LR
)
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Trainable params:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

for epoch in range(EPOCHS):
    model.train()
    start = time.perf_counter()
    loss_sum, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    print(f"Epoch {epoch+1} | Loss: {loss_sum/total:.4f} | "
          f"Accuracy: {correct/total:.4f} | "
          f"Training time: {time.perf_counter()-start:.2f}s")

## full ft

In [ ]:
import os, time, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from huggingface_hub import hf_hub_download
import torch
import torch.nn as nn
from torchvision import models

DATA_DIR = "/content/brain-tumor-mri-dataset"
BATCH_SIZE, EPOCHS, LR = 32, 10, 1e-4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/Training", transform)
test_ds  = datasets.ImageFolder(f"{DATA_DIR}/Testing", transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

model.classifier = nn.Linear(model.classifier.in_features, 4)
model = model.to(device)

print("Classifier:", model.classifier)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
criterion = nn.CrossEntropyLoss()

print("Device:", device)
print("Train samples:", len(train_ds), "| Test samples:", len(test_ds))
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

for epoch in range(EPOCHS):
    model.train()
    start = time.perf_counter()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    train_seconds = time.perf_counter() - start

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/total:.4f} | "
        f"Accuracy: {correct/total:.4f} | "
        f"Training time: {train_seconds:.2f}s"
    )